# META — Short Mejorado con shorts adicionales protegidos

Este cuaderno conserva el sistema **META - SHORT MEJORADO** original y añade únicamente shorts de continuación en los huecos donde el sistema estaba en liquidez.

La mejora busca una ruptura del mínimo de 10 sesiones después de un rebote reciente sobre EMA10, siempre dentro de tendencia bajista (`Close < SMA50` y `SMA20 < SMA50`) y con volumen superior a su media. Es una condición más rápida que la ruptura original de 20 sesiones.

Regla esencial: una señal LONG o SHORT original siempre tiene prioridad. Si aparece mientras existe un short EXTRA, este se cierra en la apertura y la operación original comienza en esa misma apertura. Al final se audita automáticamente que todas las operaciones originales mantengan tipo, sistema, fechas y precios.

In [ ]:
# Si fuera necesario: %pip install -q yfinance python-dateutil matplotlib pandas numpy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from dateutil.relativedelta import relativedelta
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

TICKER = 'META'
CAPITAL_INICIAL = 10_000
VENTANAS = [3, 6, 9, 12, 15, 18]
STOP_LONG = 0.07
STOP_SHORT = 0.07
PERIODO_RUPTURA = 20
MULTIPLICADOR_VOLUMEN = 1.00

# Parámetros exclusivos del módulo EXTRA (no alteran el original)
EXTRA_RUPTURA = 10
EXTRA_REBOTE_LOOKBACK = 5
EXTRA_STOP = 0.05
EXTRA_ATR = 2.0
EXTRA_MAX_DIAS = 15

datos = yf.download(TICKER, period='5y', interval='1d', auto_adjust=False, progress=False)
if datos.empty:
    raise RuntimeError('No se pudieron descargar los datos de META.')
if isinstance(datos.columns, pd.MultiIndex):
    datos.columns = datos.columns.get_level_values(0)
datos = datos[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
print(f'Datos: {datos.index.min().date()} → {datos.index.max().date()} | {len(datos)} sesiones')

In [ ]:
def preparar_senales(df):
    d = df.copy()
    delta = d['Close'].diff()
    subida, bajada = delta.clip(lower=0), -delta.clip(upper=0)
    rs = subida.rolling(14).mean() / bajada.rolling(14).mean()
    d['RSI'] = 100 - 100 / (1 + rs)
    d['BB_MIDDLE'] = d['Close'].rolling(20).mean()
    desv = d['Close'].rolling(20).std()
    d['BB_LOWER'] = d['BB_MIDDLE'] - 2 * desv
    d['EMA10'] = d['Close'].ewm(span=10, adjust=False).mean()
    d['SMA20'] = d['Close'].rolling(20).mean()
    d['SMA50'] = d['Close'].rolling(50).mean()
    previo = d['Close'].shift(1)
    tr = pd.concat([d['High'] - d['Low'], (d['High'] - previo).abs(), (d['Low'] - previo).abs()], axis=1).max(axis=1)
    d['ATR14'] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    # Reglas originales, sin cambios
    d['LONG_ENTRY'] = (d['RSI'] < 30) & (d['Close'] <= d['BB_LOWER'])
    d['LONG_EXIT'] = (d['RSI'] >= 65) | (d['Close'] >= d['BB_MIDDLE'])
    d['MINIMO_20_PREV'] = d['Low'].rolling(PERIODO_RUPTURA).min().shift(1)
    d['VOL_20_PREV'] = d['Volume'].rolling(PERIODO_RUPTURA).mean().shift(1)
    d['SHORT_RSI'] = d['RSI'] > 70
    d['SHORT_RUPTURA'] = (d['Close'] < d['MINIMO_20_PREV']) & (d['Volume'] > d['VOL_20_PREV'] * MULTIPLICADOR_VOLUMEN)
    d['SHORT_ENTRY'] = d['SHORT_RSI'] | d['SHORT_RUPTURA']
    d['SHORT_EXIT'] = d['RSI'] < 50

    # Candidato EXTRA: rebote reciente y nueva ruptura más rápida dentro de tendencia bajista
    d['MINIMO_EXTRA_PREV'] = d['Low'].rolling(EXTRA_RUPTURA).min().shift(1)
    d['REBOTE_RECIENTE'] = (d['Close'] > d['EMA10']).rolling(EXTRA_REBOTE_LOOKBACK).max().shift(1).fillna(False).astype(bool)
    d['SHORT_EXTRA'] = (d['Close'] < d['MINIMO_EXTRA_PREV']) & d['REBOTE_RECIENTE'] & \
                       (d['Close'] < d['SMA50']) & (d['SMA20'] < d['SMA50']) & \
                       (d['Volume'] > d['VOL_20_PREV']) & (~d['SHORT_ENTRY'])
    return d

datos = preparar_senales(datos)

In [ ]:
def ejecutar_backtest(df, permitir_extras=False):
    d = df.copy()
    capital = float(CAPITAL_INICIAL)
    posicion = precio_entrada = fecha_entrada = sistema = None
    operaciones, equity = [], []
    minimo_extra, trailing_extra, dias_extra = np.inf, np.inf, 0

    def abrir(tipo, sist, fecha, precio):
        nonlocal posicion, sistema, fecha_entrada, precio_entrada
        nonlocal minimo_extra, trailing_extra, dias_extra
        posicion, sistema, fecha_entrada, precio_entrada = tipo, sist, fecha, precio
        minimo_extra, trailing_extra, dias_extra = precio, precio * (1 + EXTRA_STOP), 0

    def cerrar(fecha, precio, motivo):
        nonlocal capital, posicion, sistema, fecha_entrada, precio_entrada
        retorno = ((precio - precio_entrada) / precio_entrada if posicion == 'LONG' else (precio_entrada - precio) / precio_entrada)
        capital *= 1 + retorno
        operaciones.append({'Tipo': posicion.replace('_BASE', '').replace('_EXTRA', ''),
            'Sistema entrada': sistema, 'Entrada': fecha_entrada, 'Precio entrada': precio_entrada,
            'Salida': fecha, 'Precio salida': precio, 'Rentabilidad %': retorno * 100, 'Motivo': motivo,
            'Es original': sistema != 'EXTRA CONTINUACIÓN'})
        posicion = sistema = fecha_entrada = precio_entrada = None

    for i in range(1, len(d)):
        fecha, fila, senal = d.index[i], d.iloc[i], d.iloc[i - 1]
        apertura, maximo, minimo, cierre = map(float, [fila['Open'], fila['High'], fila['Low'], fila['Close']])
        long_original = bool(senal['LONG_ENTRY'])
        short_original = bool(senal['SHORT_ENTRY'])

        # Un EXTRA jamás bloquea una señal original: se cierra y se abre la original en la misma apertura.
        if posicion == 'SHORT_EXTRA' and (long_original or short_original):
            cerrar(fecha, apertura, 'PRIORIDAD A SEÑAL ORIGINAL')
            if long_original:
                abrir('LONG', 'RSI + BOLLINGER', fecha, apertura)
            else:
                sist = 'RUPTURA' if bool(senal['SHORT_RUPTURA']) else 'RSI > 70'
                abrir('SHORT_BASE', sist, fecha, apertura)

        elif posicion is None:
            if long_original:
                abrir('LONG', 'RSI + BOLLINGER', fecha, apertura)
            elif short_original:
                sist = 'RUPTURA' if bool(senal['SHORT_RUPTURA']) else 'RSI > 70'
                abrir('SHORT_BASE', sist, fecha, apertura)
            elif permitir_extras and bool(senal['SHORT_EXTRA']):
                abrir('SHORT_EXTRA', 'EXTRA CONTINUACIÓN', fecha, apertura)

        elif posicion == 'LONG':
            stop = precio_entrada * (1 - STOP_LONG)
            if minimo <= stop:
                cerrar(fecha, stop, 'STOP LOSS LONG')
            elif bool(senal['LONG_EXIT']):
                cerrar(fecha, apertura, 'SEÑAL SALIDA LONG')

        elif posicion == 'SHORT_BASE':
            stop = precio_entrada * (1 + STOP_SHORT)
            if maximo >= stop:
                cerrar(fecha, stop, 'STOP LOSS SHORT')
            elif bool(senal['SHORT_EXIT']):
                cerrar(fecha, apertura, 'SEÑAL SALIDA SHORT')

        elif posicion == 'SHORT_EXTRA':
            dias_extra += 1
            stop = min(precio_entrada * (1 + EXTRA_STOP), trailing_extra)
            dos_cierres_sobre_ema = i >= 2 and bool((d['Close'].iloc[i-2:i] > d['EMA10'].iloc[i-2:i]).all())
            if maximo >= stop:
                cerrar(fecha, stop, 'STOP/TRAILING EXTRA')
            elif bool(senal['RSI'] < 35):
                cerrar(fecha, apertura, 'OBJETIVO RSI EXTRA')
            elif dos_cierres_sobre_ema:
                cerrar(fecha, apertura, 'RECUPERA EMA10')
            elif dias_extra >= EXTRA_MAX_DIAS:
                cerrar(fecha, apertura, 'TIEMPO MÁXIMO EXTRA')
            else:
                minimo_extra = min(minimo_extra, cierre)
                if pd.notna(fila['ATR14']):
                    trailing_extra = min(trailing_extra, minimo_extra + EXTRA_ATR * float(fila['ATR14']))

        if posicion == 'LONG':
            valor = capital * (1 + (cierre - precio_entrada) / precio_entrada)
        elif posicion in ('SHORT_BASE', 'SHORT_EXTRA'):
            valor = capital * (1 + (precio_entrada - cierre) / precio_entrada)
        else:
            valor = capital
        equity.append({'Fecha': fecha, 'Equity': valor})

    if posicion is not None:
        cerrar(d.index[-1], float(d['Close'].iloc[-1]), 'FIN DEL PERIODO')
        equity[-1]['Equity'] = capital
    eq = pd.DataFrame(equity).set_index('Fecha')
    dd = ((eq['Equity'] / eq['Equity'].cummax()) - 1).min() * 100
    aciertos = 100 * sum(o['Rentabilidad %'] > 0 for o in operaciones) / len(operaciones) if operaciones else 0
    return {'Capital final': capital, 'Rentabilidad %': (capital / CAPITAL_INICIAL - 1) * 100,
            'Drawdown máximo %': dd, 'Operaciones': operaciones, '% Aciertos': aciertos, 'Equity': eq}


In [ ]:
def firma_original(ops):
    return [(o['Tipo'], o['Sistema entrada'], o['Entrada'], round(o['Precio entrada'], 6),
             o['Salida'], round(o['Precio salida'], 6)) for o in ops if o['Es original']]

resultados, filas = {}, []
for meses in VENTANAS:
    periodo = datos.loc[datos.index >= datos.index[-1] - relativedelta(months=meses)].copy()
    base = ejecutar_backtest(periodo, permitir_extras=False)
    mejorado = ejecutar_backtest(periodo, permitir_extras=True)
    # Esta aserción impide aceptar una mejora que cambie una operación anterior.
    assert firma_original(base['Operaciones']) == firma_original(mejorado['Operaciones']), \
        f'La extensión alteró operaciones originales en {meses} meses.'
    resultados[meses] = {'datos': periodo, 'Original': base, 'Original + extras': mejorado}
    for nombre, r in [('Original', base), ('Original + extras', mejorado)]:
        extras = [o for o in r['Operaciones'] if not o['Es original']]
        filas.append({'Ventana': meses, 'Estrategia': nombre, 'Capital final': r['Capital final'],
            'Rentabilidad %': r['Rentabilidad %'], 'Drawdown máximo %': r['Drawdown máximo %'],
            'Operaciones totales': len(r['Operaciones']), 'Shorts extra': len(extras),
            'Rentabilidad extras %': sum(o['Rentabilidad %'] for o in extras), '% Aciertos': r['% Aciertos']})

tabla = pd.DataFrame(filas)
print('AUDITORÍA SUPERADA: las operaciones LONG y SHORT originales son idénticas en las seis ventanas.')
display(tabla.style.format({'Capital final': '${:,.2f}', 'Rentabilidad %': '{:+.2f}%',
    'Drawdown máximo %': '{:+.2f}%', 'Rentabilidad extras %': '{:+.2f}%', '% Aciertos': '{:.1f}%'}))

comparacion = tabla.pivot(index='Ventana', columns='Estrategia', values='Rentabilidad %')
comparacion['Mejora extras %'] = comparacion['Original + extras'] - comparacion['Original']
print('MEJORA NETA POR VENTANA')
display(comparacion.style.format('{:+.2f}%'))

In [ ]:
# Gráfico de 9 meses: conserva todas las operaciones originales y destaca únicamente las añadidas.
pack, periodo = resultados[9]['Original + extras'], resultados[9]['datos']
fig, ax = plt.subplots(figsize=(17, 8))
ax.plot(periodo.index, periodo['Close'], lw=1.4, color='steelblue', label='META - Cierre')
for n, op in enumerate(pack['Operaciones'], 1):
    extra = not op['Es original']
    if extra:
        color, marker = 'black', 'v'
    elif op['Tipo'] == 'LONG':
        color, marker = 'green', '^'
    elif op['Sistema entrada'] == 'RUPTURA':
        color, marker = 'purple', 'v'
    else:
        color, marker = 'red', 'v'
    ax.scatter(op['Entrada'], op['Precio entrada'], color=color, marker=marker, s=145, zorder=6)
    ax.scatter(op['Salida'], op['Precio salida'], color='gold' if extra else ('lime' if op['Tipo']=='LONG' else 'orange'),
               marker='^' if op['Tipo']=='SHORT' else 'v', s=115, zorder=6)
    ax.plot([op['Entrada'], op['Salida']], [op['Precio entrada'], op['Precio salida']], color=color, ls='--', alpha=.55)
    ax.annotate(str(n), (op['Entrada'], op['Precio entrada']), xytext=(5, 6), textcoords='offset points', color=color, weight='bold')
ax.scatter([], [], color='green', marker='^', s=90, label='LONG original')
ax.scatter([], [], color='red', marker='v', s=90, label='SHORT RSI original')
ax.scatter([], [], color='purple', marker='v', s=90, label='SHORT ruptura original')
ax.scatter([], [], color='black', marker='v', s=90, label='SHORT EXTRA continuación')
ax.set_title('META - SHORT MEJORADO ORIGINAL + EXTRAS - Últimos 9 meses')
ax.set_xlabel('Fecha'); ax.set_ylabel('Precio META ($)'); ax.grid(alpha=.25); ax.legend(loc='best')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

print('OPERACIONES DE 9 MESES (las añadidas aparecen como EXTRA CONTINUACIÓN)')
display(pd.DataFrame(pack['Operaciones']).style.format({'Precio entrada': '${:,.2f}',
    'Precio salida': '${:,.2f}', 'Rentabilidad %': '{:+.2f}%'}))